# TTD planner check (V5)

Each question has one runnable cell. It displays the question and gold joins/result as Markdown, then shows the BioChirp route and optional GPT-6 Luna review. This checks join plans; it does not execute SQL.

The local question JSON also contains the saved planner filters and endpoint. The planner-run report is provenance only; the notebook does not load it. Only the shared TTD schema is loaded separately from `evaluation/`. Each question cell makes a fresh planner call.

Run the setup cells from top to bottom, then run any question cell. Set `OPENAI_API_KEY` to enable Luna.


## TTD schema and join graph

The full table and column schema is loaded from `evaluation/schema_kg/inputs/ttd/schema.json`. Arrows show configured ID joins; they describe database links, not biological causation.

```mermaid
graph LR
  BM[biomarker_master] -- biomarker_id --> BDA[biomarker_disease]
  BDA -- disease_id --> DIS[disease_master]
  DM[drug_master] -- drug_id --> DC[drug_crossmatching]
  DM -- drug_id --> DDA[drug_disease]
  DDA -- disease_id --> DIS
  DM -- drug_id --> DS[drug_synonyms]
  DM -- drug_id --> DTA[drug_target]
  DTA -- target_id --> TM[target_master]
  TM -- target_id --> TDA[target_disease]
  TDA -- disease_id --> DIS
  TM -- target_id --> TPA[target_pathway]
  TPA -- pathway_id --> PM[pathway_master]
  TM -- target_id --> TU[target_uniprot]
  TM -- target_id --> TCA[target_compound_activity]
```



## 1. Load the benchmark and schema

In [2]:
from pathlib import Path
import json
import os
import urllib.request
from typing import Any, Optional
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Find the repository root from the notebook kernel's current folder.
BENCHMARK_DATA_RELATIVE_PATH = Path(
    "evaluation/planner/ttd_planner_check_questions_v5.json"
)
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / BENCHMARK_DATA_RELATIVE_PATH).exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError("Could not find the BioChirp repository root")
    REPO_ROOT = REPO_ROOT.parent

TTD_SCHEMA_RELATIVE_PATH = Path("evaluation/schema_kg/inputs/ttd/schema.json")
load_dotenv(REPO_ROOT / ".env")
BENCHMARK_DATA_PATH = REPO_ROOT / BENCHMARK_DATA_RELATIVE_PATH
TTD_SCHEMA_PATH = REPO_ROOT / TTD_SCHEMA_RELATIVE_PATH

benchmark_data = json.loads(BENCHMARK_DATA_PATH.read_text(encoding="utf-8"))
saved_planner_inputs = benchmark_data["saved_planner_inputs"]
TTD_SCHEMA_BY_TABLE = json.loads(TTD_SCHEMA_PATH.read_text(encoding="utf-8"))["ttd"]
questions_by_id = {item["id"]: item for item in benchmark_data["questions"]}
saved_filter_values_by_question = saved_planner_inputs["parsed_value_by_question"]
BIOCHIRP_PLANNER_URL = saved_planner_inputs["endpoint"]


## 2. Prepare planner inputs

In [3]:
# Reuse saved filter values and the question's fully qualified output columns.
assert set(questions_by_id) == set(saved_filter_values_by_question), "Question IDs differ from saved filters"


def planner_request_payload(question_id: str) -> dict[str, Any]:
    """Build the saved-input request for one benchmark question.

    Args:
        question_id (str): ID used to look up the question and its filter values.
    Returns:
        dict[str, Any]: Payload containing requested columns, filters, and wording.
    """
    question = questions_by_id[question_id]
    return {
        "kept": [[f"ttd.{column}", 1.0] for column in question["target_column_key"]],
        "parsed_value": saved_filter_values_by_question[question_id],
        "question": question["question"],
    }


def expert_tables(edges: list[tuple[str, str]]) -> set[str]:
    """Get table names appearing in fully qualified join edges.

    Args:
        edges (list[tuple[str, str]]): Join edges as pairs of table.column names.
    Returns:
        set[str]: Distinct table names used by the edges.
    """
    return {column.rsplit(".", 1)[0] for edge in edges for column in edge}


## 3. Read configured joins

In [4]:
import re

FK_PATTERN = re.compile(r"FK to ([A-Za-z0-9_]+)\.([A-Za-z0-9_]+)")


def actual_table_name(name: str) -> str:
    """Resolve a schema FK table label to its actual schema key.

    Args:
        name (str): Table name written in an FK description.
    Returns:
        str: Matching schema table name, including a suffix if required.
    """
    if name in TTD_SCHEMA_BY_TABLE:
        return name
    if name + "_ttd" in TTD_SCHEMA_BY_TABLE:
        return name + "_ttd"
    return name


# Read configured foreign keys directly from the TTD schema.
SCHEMA_FK_EDGES = []
for table, columns in TTD_SCHEMA_BY_TABLE.items():
    for column, description in columns.items():
        match = FK_PATTERN.search(description)
        if match:
            SCHEMA_FK_EDGES.append((
                f"{table}.{column}",
                f"{actual_table_name(name=match.group(1))}.{match.group(2)}",
            ))
SCHEMA_FK_EDGES.sort()


### Compare equivalent join routes

In [5]:
def key_classes(edges: list[tuple[str, str]]) -> list[list[str]]:
    """Group qualified columns connected by join equalities.

    Args:
        edges (list[tuple[str, str]]): Join edges as pairs of table.column names.
    Returns:
        list[list[str]]: Sorted connected components of equivalent ID columns.
    """
    parent = {}

    def find(value: str) -> str:
        """Find the representative column for one equality group.

        Args:
            value (str): Fully qualified column name.
        Returns:
            str: Representative column name for its group.
        """
        parent.setdefault(value, value)
        if parent[value] != value:
            parent[value] = find(value=parent[value])
        return parent[value]

    for left, right in edges:
        left_root, right_root = find(value=left), find(value=right)
        if left_root != right_root:
            parent[right_root] = left_root

    groups = {}
    for column in parent:
        groups.setdefault(find(value=column), []).append(column)
    return sorted(sorted(group) for group in groups.values())


SCHEMA_ID_EQUIVALENCE_CLASSES = key_classes(edges=SCHEMA_FK_EDGES)
ALLOWED_JOIN_EDGES = {tuple(sorted(edge)) for edge in SCHEMA_FK_EDGES}
for group in SCHEMA_ID_EQUIVALENCE_CLASSES:
    for index, left in enumerate(group):
        for right in group[index + 1:]:
            ALLOWED_JOIN_EDGES.add(tuple(sorted((left, right))))


def route_edges(join_path: Optional[list[tuple[str, str, str]]]) -> list[tuple[str, str]]:
    """Convert planner table-pair routes to qualified column-pair edges.

    Args:
        join_path (Optional[list[tuple[str, str, str]]]): Planner triples of left
            table, right table, and shared column name; may be None.
    Returns:
        list[tuple[str, str]]: Edges represented as fully qualified columns.
    """
    return [(f"{left}.{key}", f"{right}.{key}") for left, right, key in (join_path or [])]


def show_route(join_path: Optional[list[tuple[str, str, str]]]) -> None:
    """Print a planner route as readable table.column equalities.

    Args:
        join_path (Optional[list[tuple[str, str, str]]]): Planner join triples;
            may be None or empty.
    Returns:
        None: Writes the formatted route to notebook output.
    """
    if not join_path:
        print("  (no route returned)")
    for left, right, key in join_path or []:
        print(f"  {left}.{key} ↔ {right}.{key}")


### Compare a route with the expert plan


In [6]:
def compare_with_expert(
    question_id: str, tables: set[str], edges: list[tuple[str, str]]
) -> tuple[bool, bool]:
    """Compare a planner route with the question's expert join plan.

    Args:
        question_id (str): Benchmark question ID.
        tables (set[str]): Tables returned by the planner.
        edges (list[tuple[str, str]]): Planner joins as qualified column pairs.
    Returns:
        tuple[bool, bool]: Exact edge match, then expert-equivalent plan match.
    """
    question = questions_by_id[question_id]
    expert_edges = [tuple(edge.split(" ↔ ")) for edge in question["expert_join_edges"]]
    exact = {tuple(sorted(edge)) for edge in edges} == {
        tuple(sorted(edge)) for edge in expert_edges
    }
    equivalent = (
        {actual_table_name(name=table) for table in tables}
        == {actual_table_name(name=table) for table in expert_tables(edges=expert_edges)}
        and key_classes(edges=edges) == key_classes(edges=expert_edges)
    )
    return exact, equivalent


### Shared JSON request helper


In [7]:
def post_json(
    url: str, body: dict[str, Any], headers: dict[str, str], timeout: int = 30
) -> Any:
    """POST JSON data and decode the JSON response.

    Args:
        url (str): Endpoint URL.
        body (dict[str, Any]): Request body to encode as JSON.
        headers (dict[str, str]): HTTP headers for the request.
        timeout (int): Maximum wait time in seconds.
    Returns:
        Any: JSON value returned by the endpoint.
    """
    request = urllib.request.Request(
        url=url,
        data=json.dumps(body).encode("utf-8"),
        headers=headers,
        method="POST",
    )
    with urllib.request.urlopen(url=request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


## 4. Call BioChirp

In [8]:
def check_plan(question_id: str) -> dict[str, Any]:
    """Run BioChirp's planner and print its route comparison.

    Args:
        question_id (str): Benchmark question ID to send to BioChirp.
    Returns:
        dict[str, Any]: Full planner response for optional review by Luna.
    """
    result = post_json(
        url=BIOCHIRP_PLANNER_URL,
        body=planner_request_payload(question_id=question_id),
        headers={"Content-Type": "application/json"},
    )

    plan = result.get("plan") or {}
    join_path = plan.get("join_path") or []
    tables = set(plan.get("plan_tables") or [])
    edges = route_edges(join_path=join_path)
    exact_match, equivalent_match = compare_with_expert(
        question_id=question_id, tables=tables, edges=edges
    )

    print("BioChirp matched:", result.get("matched"))
    print("Tables:", ", ".join(sorted(tables)))
    print("BioChirp joins:")
    show_route(join_path=join_path)
    print("Exact expert edges:", "MATCH" if exact_match else "DIFFER")
    print("Expert-equivalent plan:", "MATCH" if equivalent_match else "DIFFER")
    return result


## 5. GPT-6 Luna API

In [9]:
OPENAI_CHAT_URL = "https://api.openai.com/v1/chat/completions"
LUNA_MODEL = "gpt-6-luna"


def ask_luna(system_prompt: str, user_data: dict[str, Any]) -> dict[str, Any]:
    """Ask GPT-6 Luna to review a planner route and parse its JSON reply.

    Args:
        system_prompt (str): Instructions for the review.
        user_data (dict[str, Any]): Question, schema, filters, and current plan.
    Returns:
        dict[str, Any]: Luna's structured review decision and explanation.
    """
    answer = post_json(
        url=OPENAI_CHAT_URL,
        body={
            "model": LUNA_MODEL,
            "reasoning_effort": "none",
            "max_completion_tokens": 2000,
            "response_format": {"type": "json_object"},
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": json.dumps(user_data, ensure_ascii=False)},
            ],
        },
        headers={
            "Authorization": f"Bearer {os.getenv('OPENAI_API_KEY')}",
            "Content-Type": "application/json",
        },
        timeout=60,
    )
    return json.loads(answer["choices"][0]["message"]["content"])


### Validate a replacement plan

In [10]:
def validate_luna_plan(
    plan: dict[str, Any], request_payload: dict[str, Any]
) -> tuple[bool, list[str], set[str], list[tuple[str, str]]]:
    """Check that Luna's replacement plan uses known, connected TTD joins.

    Args:
        plan (dict[str, Any]): Luna's proposed tables and join edges.
        request_payload (dict[str, Any]): Original request with required columns.
    Returns:
        tuple: Validity, validation errors, normalized tables, and join edges.
    """
    tables = {actual_table_name(name=str(table)) for table in (plan.get("tables") or [])}
    errors = []
    if not tables or not tables.issubset(TTD_SCHEMA_BY_TABLE):
        errors.append("Plan has no tables or names a table outside the TTD schema.")

    requested_tables = {
        item[0].split(".")[1]
        for item in request_payload["kept"]
        if item and len(item[0].split(".")) >= 3
    }
    if not requested_tables.issubset(tables):
        errors.append("Plan omits a table that owns a requested output column.")

    edges, neighbors = [], {table: set() for table in tables}
    for edge in plan.get("join_edges") or []:
        try:
            left = f"{actual_table_name(name=str(edge['left_table']))}.{edge['left_column']}"
            right = f"{actual_table_name(name=str(edge['right_table']))}.{edge['right_column']}"
        except (KeyError, TypeError):
            errors.append("A join edge has the wrong format.")
            continue
        if tuple(sorted((left, right))) not in ALLOWED_JOIN_EDGES:
            errors.append(f"Join is not supported by the schema: {left} ↔ {right}")
        left_table, right_table = left.rsplit(".", 1)[0], right.rsplit(".", 1)[0]
        if left_table not in tables or right_table not in tables:
            errors.append("A join uses a table missing from the plan.")
        neighbors.setdefault(left_table, set()).add(right_table)
        neighbors.setdefault(right_table, set()).add(left_table)
        edges.append((left, right))

    if tables:
        reached, pending = set(), [next(iter(tables))]
        while pending:
            table = pending.pop()
            if table not in reached:
                reached.add(table)
                pending.extend(neighbors.get(table, set()) - reached)
        if reached != tables:
            errors.append("The proposed joins do not connect all plan tables.")
    return not errors, errors, tables, edges


### Build the review request

In [11]:
LUNA_PROMPT = """Review the join plan using the question and supplied TTD schema. Decide whether it connects the facts through the relationships the question asks about. Do not use a gold answer or question-specific rules. If correct, return decision='keep'. Otherwise return decision='replan' with the smallest connected plan containing all requested output tables. Use only supplied foreign-key or same-ID-equivalent joins.

Return JSON: {"decision":"keep" or "replan","reason":"short explanation","tables":["schema table names"],"join_edges":[{"left_table":"...","left_column":"...","right_table":"...","right_column":"..."}]}"""


def luna_review_input(question_id: str, planner_result: dict[str, Any]) -> dict[str, Any]:
    """Build the context Luna needs to review one planner result.

    Args:
        question_id (str): Benchmark question ID.
        planner_result (dict[str, Any]): Full BioChirp planner response.
    Returns:
        dict[str, Any]: Question, schema, filters, and proposed joins for review.
    """
    question = questions_by_id[question_id]
    payload = planner_request_payload(question_id=question_id)
    plan = planner_result.get("plan") or {}
    path = plan.get("join_path") or []
    return {
        "question": question["question"],
        "answer_columns": question["target_columns"],
        "requested_columns": [item[0] for item in payload["kept"]],
        "filters": payload["parsed_value"],
        "schema": TTD_SCHEMA_BY_TABLE,
        "foreign_key_edges": SCHEMA_FK_EDGES,
        "same_ID_equivalence_classes": SCHEMA_ID_EQUIVALENCE_CLASSES,
        "current_plan": {
            "tables": plan.get("plan_tables") or [],
            "join_edges": [
                {"left_table": left, "left_column": key,
                 "right_table": right, "right_column": key}
                for left, right, key in path
            ],
        },
    }


## 6. Review the planner route

### Send one review request


In [12]:
def call_luna_review(review_input: dict[str, Any]) -> Optional[dict[str, Any]]:
    """Send one review request and report Luna's decision.

    Args:
        review_input (dict[str, Any]): Review context created for Luna.
    Returns:
        Optional[dict[str, Any]]: Luna's parsed reply, or None if the call fails.
    """
    try:
        review = ask_luna(system_prompt=LUNA_PROMPT, user_data=review_input)
    except Exception as error:
        print("Luna request failed:", type(error).__name__, str(error))
        return None

    decision = str(review.get("decision", "")).lower()
    print("Luna decision:", decision or "invalid")
    print("Reason:", review.get("reason", "(no reason returned)"))
    return review


### Keep or validate a replacement route


In [13]:
def get_luna_plan(
    question_id: str, planner_result: dict[str, Any]
) -> Optional[tuple[set[str], list[tuple[str, str]]]]:
    """Keep BioChirp's plan or return a valid Luna replacement plan.

    Args:
        question_id (str): Benchmark question ID.
        planner_result (dict[str, Any]): Full BioChirp planner response.
    Returns:
        Optional[tuple]: Normalized tables and joins, or None if unavailable.
    """
    payload = planner_request_payload(question_id=question_id)
    plan = planner_result.get("plan") or {}
    planner_tables = set(plan.get("plan_tables") or [])
    planner_path = plan.get("join_path") or []

    if not os.getenv("OPENAI_API_KEY"):
        print("Luna review skipped: OPENAI_API_KEY is not set.")
        return None

    review_input = luna_review_input(question_id=question_id, planner_result=planner_result)
    for attempt in range(3):
        review = call_luna_review(review_input=review_input)
        if review is None:
            return None

        decision = str(review.get("decision", "")).lower()
        if decision == "keep":
            tables = {actual_table_name(name=table) for table in planner_tables}
            return tables, route_edges(join_path=planner_path)
        if decision != "replan":
            print("Invalid decision; keeping the BioChirp plan.")
            return None

        valid, errors, tables, edges = validate_luna_plan(
            plan=review, request_payload=payload
        )
        if valid:
            print("Luna replacement joins:")
            for left, right in edges:
                print(f"  {left} ↔ {right}")
            return tables, edges

        print("Replan rejected:", "; ".join(errors))
        if attempt < 2:
            review_input["invalid_plan"] = review
            review_input["validation_errors"] = errors
            review_input["instruction"] = "Correct the plan using only allowed tables and joins."

    print("Keeping the BioChirp plan; Luna's replacement did not pass validation.")
    return None


### Compare the reviewed route with the expert plan


In [14]:
def llm_review_plan(question_id: str, planner_result: dict[str, Any]) -> None:
    """Review a planner route with Luna and print its expert-plan comparison.

    Args:
        question_id (str): Benchmark question ID.
        planner_result (dict[str, Any]): Full BioChirp planner response.
    Returns:
        None: Prints the final edge and key-equivalence matches.
    """
    reviewed_plan = get_luna_plan(
        question_id=question_id, planner_result=planner_result
    )
    if reviewed_plan is None:
        return

    final_tables, final_edges = reviewed_plan
    exact_match, equivalent_match = compare_with_expert(
        question_id=question_id, tables=final_tables, edges=final_edges
    )

    print("Final exact expert edges:", "MATCH" if exact_match else "DIFFER")
    print("Final expert-equivalent plan:", "MATCH" if equivalent_match else "DIFFER")


## Run questions

Each cell shows its gold reference, then runs the planner check for that question.


In [15]:
display(Markdown(r"""
## ttd_updated_01

**Question:** What is the molecular formula of imatinib?

**Case:** `standard`

**Gold joins:**

- `drug_crossmatching_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`

**Gold result:**

```json
[
  {
    "drug_name": "Imatinib",
    "formula": "C29H31N7O"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_updated_01')
llm_review_plan(question_id='ttd_updated_01', planner_result=planner_result)



## ttd_updated_01

**Question:** What is the molecular formula of imatinib?

**Case:** `standard`

**Gold joins:**

- `drug_crossmatching_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`

**Gold result:**

```json
[
  {
    "drug_name": "Imatinib",
    "formula": "C29H31N7O"
  }
]
```


BioChirp matched: True
Tables: drug_crossmatching_association_ttd, drug_master_table_ttd
BioChirp joins:
  drug_master_table_ttd.drug_id ↔ drug_crossmatching_association_ttd.drug_id
Exact expert edges: MATCH
Expert-equivalent plan: MATCH
Luna decision: keep
Reason: The plan joins the drug-name table to the formula table through their shared drug_id, covering both requested outputs.
Final exact expert edges: MATCH
Final expert-equivalent plan: MATCH


In [16]:
display(Markdown(r"""
## ttd_updated_02

**Question:** What UniProt entry name is assigned to the BRAF kinase protein (serine/threonine-protein kinase B-raf)?

**Case:** `standard`

**Gold joins:**

- `target_master_table_ttd.target_id ↔ target_uniprot_association_ttd.target_id`

**Gold result:**

```json
[
  {
    "gene_symbol": "BRAF",
    "uniprot_xref": "BRAF_HUMAN"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_updated_02')
llm_review_plan(question_id='ttd_updated_02', planner_result=planner_result)



## ttd_updated_02

**Question:** What UniProt entry name is assigned to the BRAF kinase protein (serine/threonine-protein kinase B-raf)?

**Case:** `standard`

**Gold joins:**

- `target_master_table_ttd.target_id ↔ target_uniprot_association_ttd.target_id`

**Gold result:**

```json
[
  {
    "gene_symbol": "BRAF",
    "uniprot_xref": "BRAF_HUMAN"
  }
]
```


BioChirp matched: True
Tables: target_master_table_ttd, target_uniprot_association_ttd
BioChirp joins:
  target_master_table_ttd.target_id ↔ target_uniprot_association_ttd.target_id
Exact expert edges: MATCH
Expert-equivalent plan: MATCH
Luna decision: keep
Reason: The plan filters the BRAF target in the target master table and joins by target_id to retrieve its UniProt entry name.
Final exact expert edges: MATCH
Final expert-equivalent plan: MATCH


In [17]:
display(Markdown(r"""
## ttd_updated_03

**Question:** What is the full name of the HER2 biomarker associated with breast cancer?

**Case:** `standard`

**Gold joins:**

- `biomarker_disease_association_ttd.biomarker_id ↔ biomarker_master_table_ttd.biomarker_id`
- `biomarker_disease_association_ttd.disease_id ↔ disease_master_table_ttd.disease_id`

**Gold result:**

```json
[
  {
    "disease_name": "Breast cancer",
    "biomarker_name": "Erbb2 tyrosine kinase receptor (HER2)"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_updated_03')
llm_review_plan(question_id='ttd_updated_03', planner_result=planner_result)



## ttd_updated_03

**Question:** What is the full name of the HER2 biomarker associated with breast cancer?

**Case:** `standard`

**Gold joins:**

- `biomarker_disease_association_ttd.biomarker_id ↔ biomarker_master_table_ttd.biomarker_id`
- `biomarker_disease_association_ttd.disease_id ↔ disease_master_table_ttd.disease_id`

**Gold result:**

```json
[
  {
    "disease_name": "Breast cancer",
    "biomarker_name": "Erbb2 tyrosine kinase receptor (HER2)"
  }
]
```


BioChirp matched: True
Tables: biomarker_disease_association_ttd, biomarker_master_table_ttd, disease_master_table_ttd
BioChirp joins:
  biomarker_master_table_ttd.biomarker_id ↔ biomarker_disease_association_ttd.biomarker_id
  biomarker_disease_association_ttd.disease_id ↔ disease_master_table_ttd.disease_id
Exact expert edges: MATCH
Expert-equivalent plan: MATCH
Luna decision: keep
Reason: The plan connects biomarker and disease through their association table using the supplied foreign keys, and includes both requested output tables.
Final exact expert edges: MATCH
Final expert-equivalent plan: MATCH


In [18]:
display(Markdown(r"""
## ttd_updated_04

**Question:** Does TTD associate the canonical BRAF kinase protein (serine/threonine-protein kinase B-raf) with the MAPK signaling pathway? If so, provide the corresponding target and pathway names.

**Case:** `standard`

**Gold joins:**

- `target_master_table_ttd.target_id ↔ target_pathway_association_ttd.target_id`
- `target_pathway_association_ttd.pathway_id ↔ pathway_master_table_ttd.pathway_id`

**Gold result:**

```json
[
  {
    "target_name": "Serine/threonine-protein kinase B-raf (BRAF)",
    "pathway_name": "MAPK signaling pathway"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_updated_04')
llm_review_plan(question_id='ttd_updated_04', planner_result=planner_result)



## ttd_updated_04

**Question:** Does TTD associate the canonical BRAF kinase protein (serine/threonine-protein kinase B-raf) with the MAPK signaling pathway? If so, provide the corresponding target and pathway names.

**Case:** `standard`

**Gold joins:**

- `target_master_table_ttd.target_id ↔ target_pathway_association_ttd.target_id`
- `target_pathway_association_ttd.pathway_id ↔ pathway_master_table_ttd.pathway_id`

**Gold result:**

```json
[
  {
    "target_name": "Serine/threonine-protein kinase B-raf (BRAF)",
    "pathway_name": "MAPK signaling pathway"
  }
]
```


BioChirp matched: True
Tables: pathway_master_table_ttd, target_master_table_ttd, target_pathway_association_ttd
BioChirp joins:
  pathway_master_table_ttd.pathway_id ↔ target_pathway_association_ttd.pathway_id
  target_pathway_association_ttd.target_id ↔ target_master_table_ttd.target_id
Exact expert edges: MATCH
Expert-equivalent plan: MATCH
Luna decision: keep
Reason: The plan joins the requested target and pathway through their association table using the supplied foreign keys.
Final exact expert edges: MATCH
Final expert-equivalent plan: MATCH


In [19]:
display(Markdown(r"""
## ttd_updated_05

**Question:** What is the CAS Registry Number for vemurafenib?

**Case:** `standard`

**Gold joins:**

- `drug_crossmatching_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`

**Gold result:**

```json
[
  {
    "drug_name": "Vemurafenib",
    "cas_number": "CAS 918504-65-1"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_updated_05')
llm_review_plan(question_id='ttd_updated_05', planner_result=planner_result)



## ttd_updated_05

**Question:** What is the CAS Registry Number for vemurafenib?

**Case:** `standard`

**Gold joins:**

- `drug_crossmatching_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`

**Gold result:**

```json
[
  {
    "drug_name": "Vemurafenib",
    "cas_number": "CAS 918504-65-1"
  }
]
```


BioChirp matched: True
Tables: drug_crossmatching_association_ttd, drug_master_table_ttd
BioChirp joins:
  drug_master_table_ttd.drug_id ↔ drug_crossmatching_association_ttd.drug_id
Exact expert edges: MATCH
Expert-equivalent plan: MATCH
Luna decision: keep
Reason: The plan joins the drug name to the CAS number through their shared drug_id key.
Final exact expert edges: MATCH
Final expert-equivalent plan: MATCH


In [20]:
display(Markdown(r"""
## ttd_updated_06

**Question:** What mechanism of action does TTD record for gefitinib against EGFR?

**Case:** `standard`

**Gold joins:**

- `drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id`
- `drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id`

**Gold result:**

```json
[
  {
    "drug_name": "Gefitinib",
    "gene_symbol": "EGFR",
    "drug_mechanism_of_action_on_target": "Inhibitor"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_updated_06')
llm_review_plan(question_id='ttd_updated_06', planner_result=planner_result)



## ttd_updated_06

**Question:** What mechanism of action does TTD record for gefitinib against EGFR?

**Case:** `standard`

**Gold joins:**

- `drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id`
- `drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id`

**Gold result:**

```json
[
  {
    "drug_name": "Gefitinib",
    "gene_symbol": "EGFR",
    "drug_mechanism_of_action_on_target": "Inhibitor"
  }
]
```


BioChirp matched: True
Tables: drug_master_table_ttd, drug_target_association_ttd, target_master_table_ttd
BioChirp joins:
  drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id
  drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id
Exact expert edges: MATCH
Expert-equivalent plan: MATCH
Luna decision: keep
Reason: The plan connects the drug and target through the drug-target association, which contains the requested mechanism and links to the gene symbol.
Final exact expert edges: MATCH
Final expert-equivalent plan: MATCH


In [21]:
display(Markdown(r"""
## ttd_updated_07

**Question:** Which drug is associated with the synonym CGP-57148 in TTD?

**Case:** `standard`

**Gold joins:**

- `drug_master_table_ttd.drug_id ↔ drug_synonyms_association_ttd.drug_id`

**Gold result:**

```json
[
  {
    "drug_name": "Imatinib",
    "synonym": "CGP-57148"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_updated_07')
llm_review_plan(question_id='ttd_updated_07', planner_result=planner_result)



## ttd_updated_07

**Question:** Which drug is associated with the synonym CGP-57148 in TTD?

**Case:** `standard`

**Gold joins:**

- `drug_master_table_ttd.drug_id ↔ drug_synonyms_association_ttd.drug_id`

**Gold result:**

```json
[
  {
    "drug_name": "Imatinib",
    "synonym": "CGP-57148"
  }
]
```


BioChirp matched: True
Tables: drug_master_table_ttd, drug_synonyms_association_ttd
BioChirp joins:
  drug_master_table_ttd.drug_id ↔ drug_synonyms_association_ttd.drug_id
Exact expert edges: MATCH
Expert-equivalent plan: MATCH
Luna decision: keep
Reason: The synonym association is joined to the drug master by drug_id, connecting the requested synonym and drug name.
Final exact expert edges: MATCH
Final expert-equivalent plan: MATCH


In [22]:
display(Markdown(r"""
## ttd_updated_08

**Question:** How is the EGFR T790M mutant classified as a therapeutic target, and what is its UniProt entry name?

**Case:** `standard`

**Gold joins:**

- `target_master_table_ttd.target_id ↔ target_uniprot_association_ttd.target_id`

**Gold result:**

```json
[
  {
    "target_name": "EGFR T790M mutant (EGFR T790M)",
    "target_type": "Successful target",
    "uniprot_xref": "EGFR_HUMAN"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_updated_08')
llm_review_plan(question_id='ttd_updated_08', planner_result=planner_result)



## ttd_updated_08

**Question:** How is the EGFR T790M mutant classified as a therapeutic target, and what is its UniProt entry name?

**Case:** `standard`

**Gold joins:**

- `target_master_table_ttd.target_id ↔ target_uniprot_association_ttd.target_id`

**Gold result:**

```json
[
  {
    "target_name": "EGFR T790M mutant (EGFR T790M)",
    "target_type": "Successful target",
    "uniprot_xref": "EGFR_HUMAN"
  }
]
```


BioChirp matched: True
Tables: target_master_table_ttd, target_uniprot_association_ttd
BioChirp joins:
  target_uniprot_association_ttd.target_id ↔ target_master_table_ttd.target_id
Exact expert edges: MATCH
Expert-equivalent plan: MATCH
Luna decision: keep
Reason: The plan joins the target name to its UniProt association through the shared target_id, covering all requested columns.
Final exact expert edges: MATCH
Final expert-equivalent plan: MATCH


In [23]:
display(Markdown(r"""
## ttd_updated_09

**Question:** What are the molecular formula and ChEBI ID of imatinib?

**Case:** `standard`

**Gold joins:**

- `drug_crossmatching_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`

**Gold result:**

```json
[
  {
    "drug_name": "Imatinib",
    "formula": "C29H31N7O",
    "chebi_xref": "CHEBI:45783"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_updated_09')
llm_review_plan(question_id='ttd_updated_09', planner_result=planner_result)



## ttd_updated_09

**Question:** What are the molecular formula and ChEBI ID of imatinib?

**Case:** `standard`

**Gold joins:**

- `drug_crossmatching_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`

**Gold result:**

```json
[
  {
    "drug_name": "Imatinib",
    "formula": "C29H31N7O",
    "chebi_xref": "CHEBI:45783"
  }
]
```


BioChirp matched: True
Tables: drug_crossmatching_association_ttd, drug_master_table_ttd
BioChirp joins:
  drug_master_table_ttd.drug_id ↔ drug_crossmatching_association_ttd.drug_id
Exact expert edges: MATCH
Expert-equivalent plan: MATCH
Luna decision: keep
Reason: The plan connects the drug name to the formula and ChEBI cross-reference through their shared drug_id.
Final exact expert edges: MATCH
Final expert-equivalent plan: MATCH


In [24]:
display(Markdown(r"""
## ttd_updated_10

**Question:** What full target name and gene symbol does TTD record for the BRAF kinase protein (serine/threonine-protein kinase B-raf) associated with melanoma?

**Case:** `standard`

**Gold joins:**

- `target_master_table_ttd.target_id ↔ target_disease_association_ttd.target_id`
- `target_disease_association_ttd.disease_id ↔ disease_master_table_ttd.disease_id`

**Gold result:**

```json
[
  {
    "gene_symbol": "BRAF",
    "target_name": "Serine/threonine-protein kinase B-raf (BRAF)",
    "disease_name": "Melanoma"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_updated_10')
llm_review_plan(question_id='ttd_updated_10', planner_result=planner_result)



## ttd_updated_10

**Question:** What full target name and gene symbol does TTD record for the BRAF kinase protein (serine/threonine-protein kinase B-raf) associated with melanoma?

**Case:** `standard`

**Gold joins:**

- `target_master_table_ttd.target_id ↔ target_disease_association_ttd.target_id`
- `target_disease_association_ttd.disease_id ↔ disease_master_table_ttd.disease_id`

**Gold result:**

```json
[
  {
    "gene_symbol": "BRAF",
    "target_name": "Serine/threonine-protein kinase B-raf (BRAF)",
    "disease_name": "Melanoma"
  }
]
```


BioChirp matched: True
Tables: disease_master_table_ttd, target_disease_association_ttd, target_master_table_ttd
BioChirp joins:
  disease_master_table_ttd.disease_id ↔ target_disease_association_ttd.disease_id
  target_disease_association_ttd.target_id ↔ target_master_table_ttd.target_id
Exact expert edges: MATCH
Expert-equivalent plan: MATCH
Luna decision: keep
Reason: The plan connects the requested disease and target tables through their target-disease association using the supplied foreign keys.
Final exact expert edges: MATCH
Final expert-equivalent plan: MATCH


In [25]:
display(Markdown(r"""
## ttd_updated_11

**Question:** How does imatinib act on the BCR-ABL1 fusion target, and which UniProt cross-reference identifies that target?

**Case:** `standard`

**Gold joins:**

- `drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id`
- `drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id`
- `target_master_table_ttd.target_id ↔ target_uniprot_association_ttd.target_id`

**Gold result:**

```json
[
  {
    "drug_name": "Imatinib",
    "gene_symbol": "BCR-ABL1",
    "drug_mechanism_of_action_on_target": "Inhibitor",
    "uniprot_xref": "BCR_HUMAN-ABL1_HUMAN"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_updated_11')
llm_review_plan(question_id='ttd_updated_11', planner_result=planner_result)



## ttd_updated_11

**Question:** How does imatinib act on the BCR-ABL1 fusion target, and which UniProt cross-reference identifies that target?

**Case:** `standard`

**Gold joins:**

- `drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id`
- `drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id`
- `target_master_table_ttd.target_id ↔ target_uniprot_association_ttd.target_id`

**Gold result:**

```json
[
  {
    "drug_name": "Imatinib",
    "gene_symbol": "BCR-ABL1",
    "drug_mechanism_of_action_on_target": "Inhibitor",
    "uniprot_xref": "BCR_HUMAN-ABL1_HUMAN"
  }
]
```


BioChirp matched: True
Tables: drug_master_table_ttd, drug_target_association_ttd, target_master_table_ttd, target_uniprot_association_ttd
BioChirp joins:
  drug_target_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id
  drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id
  drug_target_association_ttd.target_id ↔ target_uniprot_association_ttd.target_id
Exact expert edges: DIFFER
Expert-equivalent plan: MATCH
Luna decision: keep
Reason: The plan connects the drug to its target through the drug-target association, then joins the target to its UniProt cross-reference. All requested output tables are included.
Final exact expert edges: DIFFER
Final expert-equivalent plan: MATCH


In [26]:
display(Markdown(r"""
## ttd_updated_12

**Question:** What is imatinib's approval status for chronic myeloid leukaemia, and what is its molecular formula?

**Case:** `standard`

**Gold joins:**

- `disease_master_table_ttd.disease_id ↔ drug_disease_association_ttd.disease_id`
- `drug_crossmatching_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`
- `drug_disease_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`

**Gold result:**

```json
[
  {
    "drug_name": "Imatinib",
    "disease_name": "Chronic myeloid leukaemia",
    "approval_status": "Approved",
    "formula": "C29H31N7O"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_updated_12')
llm_review_plan(question_id='ttd_updated_12', planner_result=planner_result)



## ttd_updated_12

**Question:** What is imatinib's approval status for chronic myeloid leukaemia, and what is its molecular formula?

**Case:** `standard`

**Gold joins:**

- `disease_master_table_ttd.disease_id ↔ drug_disease_association_ttd.disease_id`
- `drug_crossmatching_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`
- `drug_disease_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`

**Gold result:**

```json
[
  {
    "drug_name": "Imatinib",
    "disease_name": "Chronic myeloid leukaemia",
    "approval_status": "Approved",
    "formula": "C29H31N7O"
  }
]
```


BioChirp matched: True
Tables: disease_master_table_ttd, drug_crossmatching_association_ttd, drug_disease_association_ttd, drug_master_table_ttd
BioChirp joins:
  drug_disease_association_ttd.disease_id ↔ disease_master_table_ttd.disease_id
  drug_disease_association_ttd.drug_id ↔ drug_crossmatching_association_ttd.drug_id
  drug_crossmatching_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id
Exact expert edges: DIFFER
Expert-equivalent plan: MATCH
Luna decision: keep
Reason: The plan connects the drug-disease approval association to the disease name and to the drug, whose crossmatching row supplies the formula.
Final exact expert edges: DIFFER
Final expert-equivalent plan: MATCH


In [27]:
display(Markdown(r"""
## ttd_updated_13

**Question:** For the EGFR exon 20 insertion mutant, what are its full target name, gene symbol, target classification, and UniProt entry name?

**Case:** `standard`

**Gold joins:**

- `target_master_table_ttd.target_id ↔ target_uniprot_association_ttd.target_id`

**Gold result:**

```json
[
  {
    "gene_symbol": "EGFR",
    "target_name": "EGFR Exon20ins mutant (EGFR Exon20ins)",
    "target_type": "Successful target",
    "uniprot_xref": "EGFR_HUMAN"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_updated_13')
llm_review_plan(question_id='ttd_updated_13', planner_result=planner_result)



## ttd_updated_13

**Question:** For the EGFR exon 20 insertion mutant, what are its full target name, gene symbol, target classification, and UniProt entry name?

**Case:** `standard`

**Gold joins:**

- `target_master_table_ttd.target_id ↔ target_uniprot_association_ttd.target_id`

**Gold result:**

```json
[
  {
    "gene_symbol": "EGFR",
    "target_name": "EGFR Exon20ins mutant (EGFR Exon20ins)",
    "target_type": "Successful target",
    "uniprot_xref": "EGFR_HUMAN"
  }
]
```


BioChirp matched: True
Tables: target_master_table_ttd, target_uniprot_association_ttd
BioChirp joins:
  target_uniprot_association_ttd.target_id ↔ target_master_table_ttd.target_id
Exact expert edges: MATCH
Expert-equivalent plan: MATCH
Luna decision: keep
Reason: The plan joins the target master row to its UniProt association by target_id, covering all requested columns and the target-name filter.
Final exact expert edges: MATCH
Final expert-equivalent plan: MATCH


In [28]:
display(Markdown(r"""
## ttd_updated_14

**Question:** For vemurafenib acting on serine/threonine-protein kinase B-raf (BRAF), what mechanism category is recorded?

**Case:** `standard`

**Gold joins:**

- `drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id`
- `drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id`

**Gold result:**

```json
[
  {
    "drug_name": "Vemurafenib",
    "gene_symbol": "BRAF",
    "drug_mechanism_of_action_on_target": "Modulator",
    "target_name": "Serine/threonine-protein kinase B-raf (BRAF)"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_updated_14')
llm_review_plan(question_id='ttd_updated_14', planner_result=planner_result)



## ttd_updated_14

**Question:** For vemurafenib acting on serine/threonine-protein kinase B-raf (BRAF), what mechanism category is recorded?

**Case:** `standard`

**Gold joins:**

- `drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id`
- `drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id`

**Gold result:**

```json
[
  {
    "drug_name": "Vemurafenib",
    "gene_symbol": "BRAF",
    "drug_mechanism_of_action_on_target": "Modulator",
    "target_name": "Serine/threonine-protein kinase B-raf (BRAF)"
  }
]
```


BioChirp matched: True
Tables: drug_master_table_ttd, drug_target_association_ttd, target_master_table_ttd
BioChirp joins:
  drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id
  drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id
Exact expert edges: MATCH
Expert-equivalent plan: MATCH
Luna decision: keep
Reason: The plan connects the drug and target master tables through the drug-target association, which contains the requested mechanism category.
Final exact expert edges: MATCH
Final expert-equivalent plan: MATCH


In [29]:
display(Markdown(r"""
## ttd_updated_15

**Question:** What is the full name of erlotinib's EGFR target, and how is the drug's mechanism of action classified?

**Case:** `standard`

**Gold joins:**

- `drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id`
- `drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id`

**Gold result:**

```json
[
  {
    "drug_name": "Erlotinib",
    "gene_symbol": "EGFR",
    "drug_mechanism_of_action_on_target": "Inhibitor",
    "target_name": "Epidermal growth factor receptor (EGFR)"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_updated_15')
llm_review_plan(question_id='ttd_updated_15', planner_result=planner_result)



## ttd_updated_15

**Question:** What is the full name of erlotinib's EGFR target, and how is the drug's mechanism of action classified?

**Case:** `standard`

**Gold joins:**

- `drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id`
- `drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id`

**Gold result:**

```json
[
  {
    "drug_name": "Erlotinib",
    "gene_symbol": "EGFR",
    "drug_mechanism_of_action_on_target": "Inhibitor",
    "target_name": "Epidermal growth factor receptor (EGFR)"
  }
]
```


BioChirp matched: True
Tables: drug_master_table_ttd, drug_target_association_ttd, target_master_table_ttd
BioChirp joins:
  drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id
  drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id
Exact expert edges: MATCH
Expert-equivalent plan: MATCH
Luna decision: keep
Reason: The plan connects the drug to its target through the drug-target association, covering the requested drug name, mechanism, gene symbol, and target name.
Final exact expert edges: MATCH
Final expert-equivalent plan: MATCH


In [30]:
display(Markdown(r"""
## ttd_updated_16

**Question:** What mechanism of action does TTD report for erlotinib against EGFR, and what approval status does it report for erlotinib's non-small-cell lung cancer indication?

**Case:** `standard`

**Gold joins:**

- `disease_master_table_ttd.disease_id ↔ drug_disease_association_ttd.disease_id`
- `drug_disease_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`
- `drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id`
- `drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id`

**Gold result:**

```json
[
  {
    "drug_name": "Erlotinib",
    "gene_symbol": "EGFR",
    "disease_name": "Non-small-cell lung cancer",
    "drug_mechanism_of_action_on_target": "Inhibitor",
    "approval_status": "Approved"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_updated_16')
llm_review_plan(question_id='ttd_updated_16', planner_result=planner_result)



## ttd_updated_16

**Question:** What mechanism of action does TTD report for erlotinib against EGFR, and what approval status does it report for erlotinib's non-small-cell lung cancer indication?

**Case:** `standard`

**Gold joins:**

- `disease_master_table_ttd.disease_id ↔ drug_disease_association_ttd.disease_id`
- `drug_disease_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`
- `drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id`
- `drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id`

**Gold result:**

```json
[
  {
    "drug_name": "Erlotinib",
    "gene_symbol": "EGFR",
    "disease_name": "Non-small-cell lung cancer",
    "drug_mechanism_of_action_on_target": "Inhibitor",
    "approval_status": "Approved"
  }
]
```


BioChirp matched: True
Tables: disease_master_table_ttd, drug_disease_association_ttd, drug_master_table_ttd, drug_target_association_ttd, target_master_table_ttd
BioChirp joins:
  drug_disease_association_ttd.disease_id ↔ disease_master_table_ttd.disease_id
  drug_disease_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id
  drug_disease_association_ttd.drug_id ↔ drug_target_association_ttd.drug_id
  drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id
Exact expert edges: DIFFER
Expert-equivalent plan: MATCH
Luna decision: keep
Reason: The plan connects the drug to its disease indication and approval status, and to its EGFR target and mechanism of action through the appropriate association tables.
Final exact expert edges: DIFFER
Final expert-equivalent plan: MATCH


In [31]:
display(Markdown(r"""
## ttd_updated_17

**Question:** For vemurafenib acting on serine/threonine-protein kinase B-raf (BRAF), what mechanism category and MAPK pathway association are recorded?

**Case:** `standard`

**Gold joins:**

- `drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id`
- `drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id`
- `pathway_master_table_ttd.pathway_id ↔ target_pathway_association_ttd.pathway_id`
- `target_master_table_ttd.target_id ↔ target_pathway_association_ttd.target_id`

**Gold result:**

```json
[
  {
    "drug_name": "Vemurafenib",
    "gene_symbol": "BRAF",
    "target_name": "Serine/threonine-protein kinase B-raf (BRAF)",
    "drug_mechanism_of_action_on_target": "Modulator",
    "pathway_name": "MAPK signaling pathway"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_updated_17')
llm_review_plan(question_id='ttd_updated_17', planner_result=planner_result)



## ttd_updated_17

**Question:** For vemurafenib acting on serine/threonine-protein kinase B-raf (BRAF), what mechanism category and MAPK pathway association are recorded?

**Case:** `standard`

**Gold joins:**

- `drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id`
- `drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id`
- `pathway_master_table_ttd.pathway_id ↔ target_pathway_association_ttd.pathway_id`
- `target_master_table_ttd.target_id ↔ target_pathway_association_ttd.target_id`

**Gold result:**

```json
[
  {
    "drug_name": "Vemurafenib",
    "gene_symbol": "BRAF",
    "target_name": "Serine/threonine-protein kinase B-raf (BRAF)",
    "drug_mechanism_of_action_on_target": "Modulator",
    "pathway_name": "MAPK signaling pathway"
  }
]
```


BioChirp matched: True
Tables: drug_master_table_ttd, drug_target_association_ttd, pathway_master_table_ttd, target_master_table_ttd, target_pathway_association_ttd
BioChirp joins:
  pathway_master_table_ttd.pathway_id ↔ target_pathway_association_ttd.pathway_id
  target_pathway_association_ttd.target_id ↔ drug_target_association_ttd.target_id
  drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id
  drug_target_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id
Exact expert edges: DIFFER
Expert-equivalent plan: MATCH
Luna decision: keep
Reason: The plan connects the drug and mechanism to the target, then connects that same target to its pathway and pathway name.
Final exact expert edges: DIFFER
Final expert-equivalent plan: MATCH


In [32]:
display(Markdown(r"""
## ttd_updated_18

**Question:** For imatinib used in chronic myeloid leukaemia, what are its approval status, molecular formula, and CAS Registry Number?

**Case:** `standard`

**Gold joins:**

- `disease_master_table_ttd.disease_id ↔ drug_disease_association_ttd.disease_id`
- `drug_crossmatching_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`
- `drug_disease_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`

**Gold result:**

```json
[
  {
    "drug_name": "Imatinib",
    "disease_name": "Chronic myeloid leukaemia",
    "approval_status": "Approved",
    "formula": "C29H31N7O",
    "cas_number": "CAS 152459-95-5"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_updated_18')
llm_review_plan(question_id='ttd_updated_18', planner_result=planner_result)



## ttd_updated_18

**Question:** For imatinib used in chronic myeloid leukaemia, what are its approval status, molecular formula, and CAS Registry Number?

**Case:** `standard`

**Gold joins:**

- `disease_master_table_ttd.disease_id ↔ drug_disease_association_ttd.disease_id`
- `drug_crossmatching_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`
- `drug_disease_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`

**Gold result:**

```json
[
  {
    "drug_name": "Imatinib",
    "disease_name": "Chronic myeloid leukaemia",
    "approval_status": "Approved",
    "formula": "C29H31N7O",
    "cas_number": "CAS 152459-95-5"
  }
]
```


BioChirp matched: True
Tables: disease_master_table_ttd, drug_crossmatching_association_ttd, drug_disease_association_ttd, drug_master_table_ttd
BioChirp joins:
  drug_disease_association_ttd.disease_id ↔ disease_master_table_ttd.disease_id
  drug_disease_association_ttd.drug_id ↔ drug_crossmatching_association_ttd.drug_id
  drug_crossmatching_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id
Exact expert edges: DIFFER
Expert-equivalent plan: MATCH
Luna decision: keep
Reason: The plan connects the drug and disease through their indication association, and links crossmatching data to the same drug for formula and CAS number.
Final exact expert edges: DIFFER
Final expert-equivalent plan: MATCH


In [33]:
display(Markdown(r"""
## ttd_updated_19

**Question:** For serine/threonine-protein kinase B-raf (BRAF) associated with MAPK signaling, what target classification and UniProt entry name does TTD record?

**Case:** `standard`

**Gold joins:**

- `pathway_master_table_ttd.pathway_id ↔ target_pathway_association_ttd.pathway_id`
- `target_master_table_ttd.target_id ↔ target_pathway_association_ttd.target_id`
- `target_master_table_ttd.target_id ↔ target_uniprot_association_ttd.target_id`

**Gold result:**

```json
[
  {
    "target_name": "Serine/threonine-protein kinase B-raf (BRAF)",
    "gene_symbol": "BRAF",
    "target_type": "Successful target",
    "uniprot_xref": "BRAF_HUMAN",
    "pathway_name": "MAPK signaling pathway"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_updated_19')
llm_review_plan(question_id='ttd_updated_19', planner_result=planner_result)



## ttd_updated_19

**Question:** For serine/threonine-protein kinase B-raf (BRAF) associated with MAPK signaling, what target classification and UniProt entry name does TTD record?

**Case:** `standard`

**Gold joins:**

- `pathway_master_table_ttd.pathway_id ↔ target_pathway_association_ttd.pathway_id`
- `target_master_table_ttd.target_id ↔ target_pathway_association_ttd.target_id`
- `target_master_table_ttd.target_id ↔ target_uniprot_association_ttd.target_id`

**Gold result:**

```json
[
  {
    "target_name": "Serine/threonine-protein kinase B-raf (BRAF)",
    "gene_symbol": "BRAF",
    "target_type": "Successful target",
    "uniprot_xref": "BRAF_HUMAN",
    "pathway_name": "MAPK signaling pathway"
  }
]
```


BioChirp matched: True
Tables: pathway_master_table_ttd, target_master_table_ttd, target_pathway_association_ttd, target_uniprot_association_ttd
BioChirp joins:
  target_master_table_ttd.target_id ↔ target_uniprot_association_ttd.target_id
  target_master_table_ttd.target_id ↔ target_pathway_association_ttd.target_id
  target_pathway_association_ttd.pathway_id ↔ pathway_master_table_ttd.pathway_id
Exact expert edges: MATCH
Expert-equivalent plan: MATCH
Luna decision: keep
Reason: The plan connects the target to its UniProt record and to the MAPK pathway through the supplied target-ID and pathway-ID relationships.
Final exact expert edges: MATCH
Final expert-equivalent plan: MATCH


In [34]:
display(Markdown(r"""
## ttd_updated_20

**Question:** What mechanism does TTD record for alpelisib acting on PIK3CA, and what approval status does it report for the drug's breast cancer indication?

**Case:** `standard`

**Gold joins:**

- `disease_master_table_ttd.disease_id ↔ drug_disease_association_ttd.disease_id`
- `drug_disease_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`
- `drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id`
- `drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id`

**Gold result:**

```json
[
  {
    "drug_name": "Alpelisib",
    "gene_symbol": "PIK3CA",
    "disease_name": "Breast cancer",
    "drug_mechanism_of_action_on_target": "Inhibitor",
    "approval_status": "Approved"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_updated_20')
llm_review_plan(question_id='ttd_updated_20', planner_result=planner_result)



## ttd_updated_20

**Question:** What mechanism does TTD record for alpelisib acting on PIK3CA, and what approval status does it report for the drug's breast cancer indication?

**Case:** `standard`

**Gold joins:**

- `disease_master_table_ttd.disease_id ↔ drug_disease_association_ttd.disease_id`
- `drug_disease_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`
- `drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id`
- `drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id`

**Gold result:**

```json
[
  {
    "drug_name": "Alpelisib",
    "gene_symbol": "PIK3CA",
    "disease_name": "Breast cancer",
    "drug_mechanism_of_action_on_target": "Inhibitor",
    "approval_status": "Approved"
  }
]
```


BioChirp matched: True
Tables: disease_master_table_ttd, drug_disease_association_ttd, drug_master_table_ttd, drug_target_association_ttd, target_master_table_ttd
BioChirp joins:
  drug_disease_association_ttd.disease_id ↔ disease_master_table_ttd.disease_id
  drug_disease_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id
  drug_disease_association_ttd.drug_id ↔ drug_target_association_ttd.drug_id
  drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id
Exact expert edges: DIFFER
Expert-equivalent plan: MATCH
Luna decision: keep
Reason: The plan connects the drug and indication through their association tables, and connects the drug to its target and gene symbol through the target association.
Final exact expert edges: DIFFER
Final expert-equivalent plan: MATCH


In [53]:
display(Markdown(r"""
## ttd_amb_min_01

**Question:** Which HER2 target associated with breast cancer does lapatinib act on, and what is its UniProt entry name?

**Case:** `ambiguous_path`

**Gold joins:**

- `drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id`
- `drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id`
- `target_master_table_ttd.target_id ↔ target_disease_association_ttd.target_id`
- `target_disease_association_ttd.disease_id ↔ disease_master_table_ttd.disease_id`
- `target_master_table_ttd.target_id ↔ target_uniprot_association_ttd.target_id`

**Gold result:**

```json
[
  {
    "drug_name": "Lapatinib",
    "gene_symbol": "ERBB2",
    "target_name": "Erbb2 tyrosine kinase receptor (HER2)",
    "disease_name": "Breast cancer",
    "uniprot_xref": "ERBB2_HUMAN"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_amb_min_01')
llm_review_plan(question_id='ttd_amb_min_01', planner_result=planner_result)



## ttd_amb_min_01

**Question:** Which HER2 target associated with breast cancer does lapatinib act on, and what is its UniProt entry name?

**Case:** `ambiguous_path`

**Gold joins:**

- `drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id`
- `drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id`
- `target_master_table_ttd.target_id ↔ target_disease_association_ttd.target_id`
- `target_disease_association_ttd.disease_id ↔ disease_master_table_ttd.disease_id`
- `target_master_table_ttd.target_id ↔ target_uniprot_association_ttd.target_id`

**Gold result:**

```json
[
  {
    "drug_name": "Lapatinib",
    "gene_symbol": "ERBB2",
    "target_name": "Erbb2 tyrosine kinase receptor (HER2)",
    "disease_name": "Breast cancer",
    "uniprot_xref": "ERBB2_HUMAN"
  }
]
```


BioChirp matched: True
Tables: disease_master_table_ttd, drug_disease_association_ttd, drug_master_table_ttd, target_disease_association_ttd, target_master_table_ttd, target_uniprot_association_ttd
BioChirp joins:
  target_uniprot_association_ttd.target_id ↔ target_master_table_ttd.target_id
  target_master_table_ttd.target_id ↔ target_disease_association_ttd.target_id
  target_disease_association_ttd.disease_id ↔ disease_master_table_ttd.disease_id
  disease_master_table_ttd.disease_id ↔ drug_disease_association_ttd.disease_id
  drug_disease_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id
Exact expert edges: DIFFER
Expert-equivalent plan: DIFFER
Luna decision: replan
Reason: The plan connects lapatinib to breast cancer but does not establish that the drug acts on the HER2 target. Connect drug to target through drug_target_association_ttd, retaining the target-disease and UniProt links.
Luna replacement joins:
  drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_i

In [49]:
display(Markdown(r"""
## ttd_amb_min_02

**Question:** For erlotinib used in non-small-cell lung cancer, what are the full name and UniProt entry name of its EGFR target?

**Case:** `ambiguous_path`

**Gold joins:**

- `disease_master_table_ttd.disease_id ↔ drug_disease_association_ttd.disease_id`
- `drug_disease_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`
- `drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id`
- `drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id`
- `target_master_table_ttd.target_id ↔ target_uniprot_association_ttd.target_id`

**Gold result:**

```json
[
  {
    "drug_name": "Erlotinib",
    "gene_symbol": "EGFR",
    "disease_name": "Non-small-cell lung cancer",
    "target_name": "Epidermal growth factor receptor (EGFR)",
    "uniprot_xref": "EGFR_HUMAN"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_amb_min_02')
llm_review_plan(question_id='ttd_amb_min_02', planner_result=planner_result)



## ttd_amb_min_02

**Question:** For erlotinib used in non-small-cell lung cancer, what are the full name and UniProt entry name of its EGFR target?

**Case:** `ambiguous_path`

**Gold joins:**

- `disease_master_table_ttd.disease_id ↔ drug_disease_association_ttd.disease_id`
- `drug_disease_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`
- `drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id`
- `drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id`
- `target_master_table_ttd.target_id ↔ target_uniprot_association_ttd.target_id`

**Gold result:**

```json
[
  {
    "drug_name": "Erlotinib",
    "gene_symbol": "EGFR",
    "disease_name": "Non-small-cell lung cancer",
    "target_name": "Epidermal growth factor receptor (EGFR)",
    "uniprot_xref": "EGFR_HUMAN"
  }
]
```


BioChirp matched: True
Tables: disease_master_table_ttd, drug_disease_association_ttd, drug_master_table_ttd, target_disease_association_ttd, target_master_table_ttd, target_uniprot_association_ttd
BioChirp joins:
  target_uniprot_association_ttd.target_id ↔ target_master_table_ttd.target_id
  target_master_table_ttd.target_id ↔ target_disease_association_ttd.target_id
  target_disease_association_ttd.disease_id ↔ disease_master_table_ttd.disease_id
  disease_master_table_ttd.disease_id ↔ drug_disease_association_ttd.disease_id
  drug_disease_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id
Exact expert edges: DIFFER
Expert-equivalent plan: DIFFER
Luna decision: replan
Reason: The plan connects the drug to the disease and the target to the disease, but does not establish that erlotinib targets EGFR. Connect drug and target through drug_target_association_ttd instead.
Replan rejected: Plan omits a table that owns a requested output column.
Luna decision: replan
Reason: The plan l

In [37]:
display(Markdown(r"""
## ttd_amb_min_03

**Question:** For erlotinib's non-small-cell lung cancer indication, what is the full name of its EGFR drug target, and what relationship does TTD record between this target and the MAPK signaling pathway?

**Case:** `ambiguous_path`

**Gold joins:**

- `disease_master_table_ttd.disease_id ↔ drug_disease_association_ttd.disease_id`
- `drug_disease_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`
- `drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id`
- `drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id`
- `pathway_master_table_ttd.pathway_id ↔ target_pathway_association_ttd.pathway_id`
- `target_master_table_ttd.target_id ↔ target_pathway_association_ttd.target_id`

**Gold result:**

```json
[
  {
    "drug_name": "Erlotinib",
    "gene_symbol": "EGFR",
    "disease_name": "Non-small-cell lung cancer",
    "target_name": "Epidermal growth factor receptor (EGFR)",
    "pathway_name": "MAPK signaling pathway"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_amb_min_03')
llm_review_plan(question_id='ttd_amb_min_03', planner_result=planner_result)



## ttd_amb_min_03

**Question:** For erlotinib's non-small-cell lung cancer indication, what is the full name of its EGFR drug target, and what relationship does TTD record between this target and the MAPK signaling pathway?

**Case:** `ambiguous_path`

**Gold joins:**

- `disease_master_table_ttd.disease_id ↔ drug_disease_association_ttd.disease_id`
- `drug_disease_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`
- `drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id`
- `drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id`
- `pathway_master_table_ttd.pathway_id ↔ target_pathway_association_ttd.pathway_id`
- `target_master_table_ttd.target_id ↔ target_pathway_association_ttd.target_id`

**Gold result:**

```json
[
  {
    "drug_name": "Erlotinib",
    "gene_symbol": "EGFR",
    "disease_name": "Non-small-cell lung cancer",
    "target_name": "Epidermal growth factor receptor (EGFR)",
    "pathway_name": "MAPK signaling pathway"
  }
]
```


BioChirp matched: True
Tables: disease_master_table_ttd, drug_disease_association_ttd, drug_master_table_ttd, pathway_master_table_ttd, target_disease_association_ttd, target_master_table_ttd, target_pathway_association_ttd
BioChirp joins:
  drug_master_table_ttd.drug_id ↔ drug_disease_association_ttd.drug_id
  drug_disease_association_ttd.disease_id ↔ disease_master_table_ttd.disease_id
  disease_master_table_ttd.disease_id ↔ target_disease_association_ttd.disease_id
  target_disease_association_ttd.target_id ↔ target_master_table_ttd.target_id
  target_disease_association_ttd.target_id ↔ target_pathway_association_ttd.target_id
  target_pathway_association_ttd.pathway_id ↔ pathway_master_table_ttd.pathway_id
Exact expert edges: DIFFER
Expert-equivalent plan: DIFFER
Luna decision: replan
Reason: The plan connects the target and pathway through target-disease association, but the requested relationship is the target-to-pathway association. Connect drug and disease to the target via dru

In [47]:
display(Markdown(r"""
## ttd_amb_min_04

**Question:** Which drugs act on GRB2 mRNA associated with chronic myeloid leukaemia, and what mechanisms of action are reported?

**Case:** `ambiguous_path`

**Gold joins:**

- `target_master_table_ttd.target_id ↔ target_disease_association_ttd.target_id`
- `target_disease_association_ttd.disease_id ↔ disease_master_table_ttd.disease_id`
- `target_master_table_ttd.target_id ↔ drug_target_association_ttd.target_id`
- `drug_target_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`

**Gold result:**

```json
[
  {
    "drug_name": "BP-100-1-01",
    "gene_symbol": "GRB2",
    "target_name": "GRB2 messenger RNA (GRB2 mRNA)",
    "disease_name": "Chronic myeloid leukaemia",
    "drug_mechanism_of_action_on_target": "Modulator"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_amb_min_04')
llm_review_plan(question_id='ttd_amb_min_04', planner_result=planner_result)



## ttd_amb_min_04

**Question:** Which drugs act on GRB2 mRNA associated with chronic myeloid leukaemia, and what mechanisms of action are reported?

**Case:** `ambiguous_path`

**Gold joins:**

- `target_master_table_ttd.target_id ↔ target_disease_association_ttd.target_id`
- `target_disease_association_ttd.disease_id ↔ disease_master_table_ttd.disease_id`
- `target_master_table_ttd.target_id ↔ drug_target_association_ttd.target_id`
- `drug_target_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id`

**Gold result:**

```json
[
  {
    "drug_name": "BP-100-1-01",
    "gene_symbol": "GRB2",
    "target_name": "GRB2 messenger RNA (GRB2 mRNA)",
    "disease_name": "Chronic myeloid leukaemia",
    "drug_mechanism_of_action_on_target": "Modulator"
  }
]
```


BioChirp matched: True
Tables: disease_master_table_ttd, drug_disease_association_ttd, drug_master_table_ttd, drug_target_association_ttd, target_master_table_ttd
BioChirp joins:
  drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id
  drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id
  drug_master_table_ttd.drug_id ↔ drug_disease_association_ttd.drug_id
  drug_disease_association_ttd.disease_id ↔ disease_master_table_ttd.disease_id
Exact expert edges: DIFFER
Expert-equivalent plan: DIFFER
Luna decision: replan
Reason: The question asks for targets (GRB2) associated with the disease, then drugs acting on those targets. The current plan links disease only to drugs, which does not establish the target–disease relationship. Use target_disease_association_ttd instead.
Luna replacement joins:
  disease_master_table_ttd.disease_id ↔ target_disease_association_ttd.disease_id
  target_disease_association_ttd.target_id ↔ target_master_table_ttd.target_id
 

In [39]:
display(Markdown(r"""
## ttd_amb_min_05

**Question:** Which EGFR T790M target associated with non-small-cell lung cancer does osimertinib act on, and what is its UniProt entry name?

**Case:** `ambiguous_path`

**Gold joins:**

- `drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id`
- `drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id`
- `target_master_table_ttd.target_id ↔ target_disease_association_ttd.target_id`
- `target_disease_association_ttd.disease_id ↔ disease_master_table_ttd.disease_id`
- `target_master_table_ttd.target_id ↔ target_uniprot_association_ttd.target_id`

**Gold result:**

```json
[
  {
    "drug_name": "Osimertinib",
    "gene_symbol": "EGFR",
    "target_name": "EGFR T790M mutant (EGFR T790M)",
    "disease_name": "Non-small-cell lung cancer",
    "uniprot_xref": "EGFR_HUMAN"
  }
]
```
"""))

planner_result = check_plan(question_id='ttd_amb_min_05')
llm_review_plan(question_id='ttd_amb_min_05', planner_result=planner_result)



## ttd_amb_min_05

**Question:** Which EGFR T790M target associated with non-small-cell lung cancer does osimertinib act on, and what is its UniProt entry name?

**Case:** `ambiguous_path`

**Gold joins:**

- `drug_master_table_ttd.drug_id ↔ drug_target_association_ttd.drug_id`
- `drug_target_association_ttd.target_id ↔ target_master_table_ttd.target_id`
- `target_master_table_ttd.target_id ↔ target_disease_association_ttd.target_id`
- `target_disease_association_ttd.disease_id ↔ disease_master_table_ttd.disease_id`
- `target_master_table_ttd.target_id ↔ target_uniprot_association_ttd.target_id`

**Gold result:**

```json
[
  {
    "drug_name": "Osimertinib",
    "gene_symbol": "EGFR",
    "target_name": "EGFR T790M mutant (EGFR T790M)",
    "disease_name": "Non-small-cell lung cancer",
    "uniprot_xref": "EGFR_HUMAN"
  }
]
```


BioChirp matched: True
Tables: disease_master_table_ttd, drug_disease_association_ttd, drug_master_table_ttd, target_disease_association_ttd, target_master_table_ttd, target_uniprot_association_ttd
BioChirp joins:
  target_uniprot_association_ttd.target_id ↔ target_master_table_ttd.target_id
  target_master_table_ttd.target_id ↔ target_disease_association_ttd.target_id
  target_disease_association_ttd.disease_id ↔ disease_master_table_ttd.disease_id
  disease_master_table_ttd.disease_id ↔ drug_disease_association_ttd.disease_id
  drug_disease_association_ttd.drug_id ↔ drug_master_table_ttd.drug_id
Exact expert edges: DIFFER
Expert-equivalent plan: DIFFER
Luna decision: replan
Reason: The plan connects the drug to the disease, but does not establish that osimertinib acts on the EGFR T790M target. Join the drug and target through drug_target_association_ttd; the disease relationship can be retained through target_disease_association_ttd.
Luna replacement joins:
  drug_master_table_ttd.dr